# Lecture 14: Categorical Variables and Interactions

We build the regression step by step around three questions:

1. How do prices differ across cities?
2. How do hotels and apartments differ in price, holding city constant?
3. Does the price difference between star categories vary by city?

Each row in `hotels_course.csv` is one accommodation offer from the European
hotels data. This prepared extract includes hotels and apartments with 3,
3.5 or 4 stars in all three cities. We use all 821 observations.
Its preparation and price restriction are documented in `data/README.md`.

Inspect each comparison visually before estimating it. All models use the
same observations and HC0 standard errors. Keep prices and comparisons on
the log scale throughout.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

In [ ]:
hotels = pd.read_csv("../../data/raw/hotels_course.csv")
hotels.groupby(["city", "accommodation"]).size()

## 1. How do prices differ across cities?

Begin with city alone. Use **Amsterdam** as the reference category. The
intercept is Amsterdam's mean log price; the other city coefficients are
differences from that mean. No other characteristics are held constant yet.
Omitting Amsterdam's indicator does not remove its observations.

In [ ]:
sns.boxplot(data=hotels, x="city", y="log_price")
plt.title("Log price across cities")
plt.show()

In [ ]:
# Compare these means with the intercept and city coefficients below.
hotels.groupby("city")["log_price"].mean().round(3)

In [ ]:
city_model = smf.ols(
    "log_price ~ C(city, Treatment(reference='Amsterdam'))",
    data=hotels, missing="raise",
).fit(cov_type="HC0")
city_model.summary()

## 2. How do hotels and apartments differ in price?

Keep city in the model and add accommodation type, using **Apartment** as
its reference. The Hotel coefficient is now the hotel-apartment log-price
difference holding city constant. This model assumes that difference is
the same in every city; distance, rating and stars are not controlled for yet.

In [ ]:
# Count hotels and apartments separately in each city.
pd.crosstab(hotels["city"], hotels["accommodation"])

In [ ]:
sns.boxplot(data=hotels, x="accommodation", y="log_price", hue="city")
plt.legend(title="City", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("Log price by accommodation type and city")
plt.show()

In [ ]:
accommodation_model = smf.ols(
    "log_price ~ C(city, Treatment(reference='Amsterdam'))"
    " + C(accommodation, Treatment(reference='Apartment'))",
    data=hotels, missing="raise",
).fit(cov_type="HC0")
accommodation_model.summary()

How do the city coefficients change when accommodation type enters? What
does the Hotel coefficient compare with?

## 3. Do star-category comparisons differ across cities?

First inspect the star categories within each city. We will add distance,
customer rating and star categories to the model, then allow the star
comparisons to differ by city.

In [ ]:
# Count properties in each combination of city and star category.
pd.crosstab(hotels["city"], hotels["stars"])

In [ ]:
sns.boxplot(data=hotels, x="stars", y="log_price", hue="city")
plt.legend(title="City", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("Log price by stars and city")
plt.show()

Which city-star groups have few observations? Does the pattern across stars
look similar in every city?

### 3.1 Add the other characteristics, with a common star pattern

Use **3 stars**, **Amsterdam** and **Apartment** as the reference categories.
With an intercept, each categorical predictor leaves out one indicator
column; observations in that category remain in the sample.

This model allows cities to have different log-price levels but assumes
the 3.5-versus-3 and 4-versus-3 comparisons are the same across cities.
Distance and customer rating are the numeric predictors from the previous
lecture. Each star comparison holds these predictors, accommodation type
and city constant. `Treatment(reference=...)` names each reference explicitly.

In [ ]:
category_model = smf.ols(
    "log_price ~ distance_miles + rating"
    " + C(stars, Treatment(reference=3))"
    " + C(city, Treatment(reference='Amsterdam'))"
    " + C(accommodation, Treatment(reference='Apartment'))",
    data=hotels, missing="raise",
).fit(cov_type="HC0")
category_model.summary()

In [ ]:
category_terms = category_model.params.index[category_model.params.index.str.startswith("C(")]
pd.DataFrame(
    {"estimate": category_model.params[category_terms], "robust_se": category_model.bse[category_terms]}
).round(3)

What does each star coefficient compare with? How do the city coefficients
differ from the star coefficients?

The intercept refers to a 3-star apartment in Amsterdam when distance and
rating are zero. It is not an average price; zero is outside the observed
rating range.

### 3.2 Allow the star comparisons to differ by city

`*` includes both categorical predictors and their interaction. The star
coefficients now describe comparisons within **Amsterdam**. A city-by-star
interaction is the difference between that city's star comparison and
Amsterdam's comparison; add the two coefficients to obtain the comparison
for Barcelona or Vienna.

The city coefficients now compare cities specifically at **3 stars**.
Distance and rating associations remain common across cities. Accommodation type
is not included in this model.

In [ ]:
interaction_model = smf.ols(
    "log_price ~ distance_miles + rating"
    " + C(stars, Treatment(reference=3)) * C(city, Treatment(reference='Amsterdam'))",
    data=hotels, missing="raise",
).fit(cov_type="HC0")
interaction_model.summary()

## 4. What log prices does the model predict for the observed properties?

Use each property's recorded city, stars, distance and rating to predict its log
price. Compare the predictions with the observed log prices.

In [ ]:
# Use the fitted model to predict log price for each observed property.
hotels["predicted_log_price"] = interaction_model.predict(hotels)

In [ ]:
# Display observed and predicted log prices alongside the model's predictors.
hotels[
    ["city", "stars", "distance_miles", "rating", "log_price", "predicted_log_price"]
].head(10).round(3)

- For which of the displayed properties does the model predict a higher or lower
  log price than we observe?
- Why can two properties in the same city and star category have different predictions?

These predictions are the model's **fitted values**, equivalent to
`interaction_model.fittedvalues`. They describe the fit to the estimation sample;
they do not evaluate prediction performance on new data.